# AIOps W2 D1 — Alert Correlation

Pipeline:

1. Load `dataset/alerts_sample.jsonl` và `dataset/services.json`
2. Layer 1 — Dedup bằng fingerprint `service|metric|severity`
3. Layer 2 — Session Window theo `gap_sec`
4. Layer 3 — Topology-aware grouping có hướng
5. Ghi kết quả ra `results/cluster_summary.json`

Mindset chính:
- Dedup để giảm duplicate alert.
- Time-window để gom các alert xảy ra trong cùng burst.
- Topology để kiểm tra khả năng lan truyền lỗi theo service dependency.


In [73]:
from pathlib import Path
import json
from datetime import datetime
from collections import defaultdict
import networkx as nx

DATASET_DIR = Path("dataset")
RESULTS_DIR = Path("results")
ALERTS_PATH = DATASET_DIR / "alerts_sample.jsonl"
SERVICES_PATH = DATASET_DIR / "services.json"
OUTPUT_PATH = RESULTS_DIR / "cluster_summary.json"

GAP_SEC = 120
MAX_HOP = 1

print("=" * 60)
print("CONFIG")
print("=" * 60)
print("Alerts path :", ALERTS_PATH)
print("Services path:", SERVICES_PATH)
print("Output path :", OUTPUT_PATH)
print("gap_sec     :", GAP_SEC)
print("max_hop     :", MAX_HOP)


CONFIG
Alerts path : dataset/alerts_sample.jsonl
Services path: dataset/services.json
Output path : results/cluster_summary.json
gap_sec     : 120
max_hop     : 1


In [74]:
def load_jsonl(path: Path) -> list[dict]:
    alerts = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                alerts.append(json.loads(line))
    return alerts


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def parse_ts(ts: str) -> datetime:
    return datetime.fromisoformat(ts.replace("Z", "+00:00"))


def severity_rank(severity: str) -> int:
    ranks = {
        "info": 0,
        "warn": 1,
        "warning": 1,
        "crit": 2,
        "critical": 2,
    }
    return ranks.get(str(severity).lower(), 0)


def max_severity(alerts: list[dict]) -> str:
    return max((a.get("severity", "info") for a in alerts), key=severity_rank)


def build_graph(services_data: dict) -> nx.DiGraph:
    """
    Build directed dependency graph.

    Convention:
    edge A -> B means A calls/depends on B.
    If B fails, impact can propagate upstream to A.
    """
    graph = nx.DiGraph()

    for service in services_data.get("services", []):
        graph.add_node(service["name"], kind="service", **service)

    for store in services_data.get("stores", []):
        graph.add_node(store["name"], kind="store", **store)

    for edge in services_data.get("edges", []):
        src = edge.get("from")
        dst = edge.get("to")
        if src and dst:
            graph.add_edge(src, dst, **edge)

    return graph


print("Utility functions loaded")


Utility functions loaded


In [75]:
print("=" * 60)
print("LOADING DATA")
print("=" * 60)

if not ALERTS_PATH.exists():
    raise FileNotFoundError(f"Missing file: {ALERTS_PATH}")

if not SERVICES_PATH.exists():
    raise FileNotFoundError(f"Missing file: {SERVICES_PATH}")

alerts = load_jsonl(ALERTS_PATH)
services_data = load_json(SERVICES_PATH)
graph = build_graph(services_data)

print("Services:", len(services_data.get("services", [])))
print("Stores  :", len(services_data.get("stores", [])))
print("Alerts  :", len(alerts))
print("Nodes   :", graph.number_of_nodes())
print("Edges   :", graph.number_of_edges())


LOADING DATA
Services: 10
Stores  : 4
Alerts  : 20
Nodes   : 14
Edges   : 17


## Layer 1 — Dedup

Fingerprint dùng `service|metric|severity`.

Không include `timestamp` hoặc `value` vì hai field này thay đổi theo từng lần alert fire. Nếu đưa vào fingerprint thì gần như alert nào cũng khác nhau và dedup mất tác dụng.


In [76]:
def fingerprint(alert: dict) -> str:
    return f"{alert['service']}|{alert['metric']}|{alert['severity']}"


def deduplicate_alerts(alerts: list[dict]) -> list[dict]:
    groups = {}

    for alert in sorted(alerts, key=lambda a: a["ts"]):
        fp = fingerprint(alert)

        if fp not in groups:
            item = alert.copy()
            item["fingerprint"] = fp
            item["duplicate_count"] = 1
            item["alert_ids"] = [alert["id"]]
            item["first_seen"] = alert["ts"]
            item["last_seen"] = alert["ts"]
            groups[fp] = item
        else:
            groups[fp]["duplicate_count"] += 1
            groups[fp]["alert_ids"].append(alert["id"])
            groups[fp]["first_seen"] = min(groups[fp]["first_seen"], alert["ts"])
            groups[fp]["last_seen"] = max(groups[fp]["last_seen"], alert["ts"])

            # Keep the most severe representative if severity changes.
            if severity_rank(alert.get("severity")) > severity_rank(groups[fp].get("severity")):
                old_meta = {
                    "fingerprint": groups[fp]["fingerprint"],
                    "duplicate_count": groups[fp]["duplicate_count"],
                    "alert_ids": groups[fp]["alert_ids"],
                    "first_seen": groups[fp]["first_seen"],
                    "last_seen": groups[fp]["last_seen"],
                }
                groups[fp] = alert.copy()
                groups[fp].update(old_meta)

    return sorted(groups.values(), key=lambda a: a["first_seen"])


deduped_alerts = deduplicate_alerts(alerts)

print("=" * 60)
print("LAYER 1 — DEDUP")
print("=" * 60)
print("Input alerts   :", len(alerts))
print("Deduped alerts :", len(deduped_alerts))
print("Removed dupes  :", len(alerts) - len(deduped_alerts))
for a in deduped_alerts[:5]:
    print("-", a["fingerprint"], "count=", a["duplicate_count"])


LAYER 1 — DEDUP
Input alerts   : 20
Deduped alerts : 17
Removed dupes  : 3
- payment-svc|db_connection_pool_used_ratio|warn count= 1
- payment-svc|db_connection_pool_used_ratio|crit count= 2
- payment-svc|latency_p99_ms|crit count= 3
- payment-svc|error_rate|warn count= 1
- checkout-svc|latency_p99_ms|warn count= 1


## Layer 2 — Session Window

Session window gom các alert gần nhau về thời gian. Session bị ngắt khi khoảng cách giữa hai alert liên tiếp lớn hơn `gap_sec`.


In [77]:
def session_groups(alerts: list[dict], gap_sec: int = 120) -> list[list[dict]]:
    if not alerts:
        return []

    sorted_alerts = sorted(alerts, key=lambda a: parse_ts(a.get("first_seen", a["ts"])))
    groups = [[sorted_alerts[0]]]

    for alert in sorted_alerts[1:]:
        current_ts = parse_ts(alert.get("first_seen", alert["ts"]))
        last_ts = parse_ts(groups[-1][-1].get("last_seen", groups[-1][-1]["ts"]))
        gap = (current_ts - last_ts).total_seconds()

        if gap <= gap_sec:
            groups[-1].append(alert)
        else:
            groups.append([alert])

    return groups


sessions = session_groups(deduped_alerts, gap_sec=GAP_SEC)

print("=" * 60)
print("LAYER 2 — SESSION WINDOW")
print("=" * 60)
print("gap_sec :", GAP_SEC)
print("sessions:", len(sessions))
for idx, session in enumerate(sessions):
    start = min(a.get("first_seen", a["ts"]) for a in session)
    end = max(a.get("last_seen", a["ts"]) for a in session)
    print(f"session-{idx:03d}: fingerprints={len(session)}, time_range=[{start}, {end}]")


LAYER 2 — SESSION WINDOW
gap_sec : 120
sessions: 1
session-000: fingerprints=17, time_range=[2026-06-12T09:42:01Z, 2026-06-12T09:48:30Z]


## Layer 3 — Topology-aware Grouping

Logic tổng quát:

- Cùng service thì gom.
- Nếu khác service, chỉ gom khi có quan hệ dependency có hướng trong `max_hop`.
- Ưu tiên hướng lan truyền lỗi upstream:
  - Nếu `A -> B` nghĩa là A gọi B.
  - Khi B lỗi, A có thể alert theo sau.
  - Vì vậy cặp alert hợp lý khi downstream alert xuất hiện trước hoặc gần trước upstream alert.
- Metric phải có khả năng liên quan về mặt causal:
  - `db`, `latency`, `error` thường có thể đi chung trong một incident.
  - `queue`, `resource`, `other` được giữ thận trọng hơn để tránh over-correlation.


In [78]:
def metric_family(metric: str) -> str:
    metric = metric.lower()

    if "db" in metric or "connection" in metric or "pool" in metric:
        return "db"
    if "latency" in metric or "p99" in metric:
        return "latency"
    if "error" in metric or "5xx" in metric or "drop" in metric:
        return "error"
    if "queue" in metric or "lag" in metric or "depth" in metric:
        return "queue"
    if "cpu" in metric or "memory" in metric:
        return "resource"

    return "other"


def metric_compatible(a1: dict, a2: dict) -> bool:
    f1 = metric_family(a1["metric"])
    f2 = metric_family(a2["metric"])

    # Conservative causal set for synchronous incidents.
    sync_causal = {"db", "latency", "error"}

    if f1 in sync_causal and f2 in sync_causal:
        return True

    # Same family can be correlated, but avoid broad "other" merge.
    if f1 == f2 and f1 != "other":
        return True

    return False


def shortest_path_len(graph: nx.DiGraph, src: str, dst: str):
    try:
        return nx.shortest_path_length(graph, src, dst)
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return None


def topology_causal_direction(a1: dict, a2: dict, graph: nx.DiGraph, max_hop: int = 1) -> bool:
    """
    Decide whether topology supports a causal relationship.

    Edge A -> B means A depends on B.
    If B degrades, A may alert after B.

    For alert pair:
    - downstream service alert earlier and upstream later: good causal direction
    - allow small timestamp disorder because alerts are noisy
    """
    s1, s2 = a1["service"], a2["service"]
    if s1 == s2:
        return True

    t1 = parse_ts(a1.get("first_seen", a1["ts"]))
    t2 = parse_ts(a2.get("first_seen", a2["ts"]))

    # s1 depends on s2: s2 can cause s1.
    d_1_depends_2 = shortest_path_len(graph, s1, s2)

    # s2 depends on s1: s1 can cause s2.
    d_2_depends_1 = shortest_path_len(graph, s2, s1)

    allowed_disorder_sec = 30

    if d_1_depends_2 is not None and d_1_depends_2 <= max_hop:
        # s2 is downstream/root side, s1 is upstream/affected side.
        # s2 should appear before s1, allowing small disorder.
        return (t1 - t2).total_seconds() >= -allowed_disorder_sec

    if d_2_depends_1 is not None and d_2_depends_1 <= max_hop:
        # s1 is downstream/root side, s2 is upstream/affected side.
        return (t2 - t1).total_seconds() >= -allowed_disorder_sec

    return False


def should_merge_alerts(a1: dict, a2: dict, graph: nx.DiGraph, max_hop: int = 1) -> bool:
    if a1["service"] == a2["service"]:
        return True

    if not topology_causal_direction(a1, a2, graph, max_hop=max_hop):
        return False

    if not metric_compatible(a1, a2):
        return False

    return True


def topology_grouping(alerts: list[dict], graph: nx.DiGraph, max_hop: int = 1) -> list[list[dict]]:
    n = len(alerts)
    if n == 0:
        return []

    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for i in range(n):
        for j in range(i + 1, n):
            if should_merge_alerts(alerts[i], alerts[j], graph, max_hop=max_hop):
                union(i, j)

    groups = defaultdict(list)
    for i, alert in enumerate(alerts):
        groups[find(i)].append(alert)

    return list(groups.values())


print("Generic topology-aware grouping loaded")


Generic topology-aware grouping loaded


In [79]:
def summarize_cluster(cluster_id: str, grouped_alerts: list[dict]) -> dict:
    all_alert_ids = []
    for alert in grouped_alerts:
        all_alert_ids.extend(alert.get("alert_ids", [alert["id"]]))

    return {
        "cluster_id": cluster_id,
        "alert_count": sum(a.get("duplicate_count", 1) for a in grouped_alerts),
        "services": sorted({a["service"] for a in grouped_alerts}),
        "time_range": [
            min(a.get("first_seen", a["ts"]) for a in grouped_alerts),
            max(a.get("last_seen", a["ts"]) for a in grouped_alerts),
        ],
        "max_severity": max_severity(grouped_alerts),
        "fingerprints": sorted({a["fingerprint"] for a in grouped_alerts}),
        "alert_ids": sorted(all_alert_ids),
    }


def correlate(alerts: list[dict], graph: nx.DiGraph, gap_sec: int = 120, max_hop: int = 1) -> dict:
    deduped = deduplicate_alerts(alerts)
    sessions = session_groups(deduped, gap_sec=gap_sec)

    clusters = []
    for s_idx, session_alerts in enumerate(sessions):
        topology_groups = topology_grouping(session_alerts, graph, max_hop=max_hop)

        for g_idx, group in enumerate(topology_groups):
            cluster_id = f"c-{s_idx:03d}-{g_idx:03d}"
            clusters.append(summarize_cluster(cluster_id, group))

    input_alerts = len(alerts)
    output_clusters = len(clusters)
    reduction_ratio = 1 - output_clusters / input_alerts if input_alerts else 0

    return {
        "input_alerts": input_alerts,
        "output_clusters": output_clusters,
        "reduction_ratio": round(reduction_ratio, 4),
        "gap_sec": gap_sec,
        "max_hop": max_hop,
        "deduped_alerts": len(deduped),
        "clusters": clusters,
    }


result = correlate(alerts, graph, gap_sec=GAP_SEC, max_hop=MAX_HOP)

print("=" * 60)
print("RESULT")
print("=" * 60)
print("Input Alerts      :", result["input_alerts"])
print("Deduped Alerts    :", result["deduped_alerts"])
print("Output Clusters   :", result["output_clusters"])
print("Reduction Ratio   :", f'{result["reduction_ratio"]:.2%}')
print()
print("Cluster preview:")
for c in result["clusters"]:
    print(
        c["cluster_id"],
        "alerts=", c["alert_count"],
        "services=", c["services"],
        "fingerprints=", len(c["fingerprints"]),
    )


RESULT
Input Alerts      : 20
Deduped Alerts    : 17
Output Clusters   : 3
Reduction Ratio   : 85.00%

Cluster preview:
c-000-000 alerts= 17 services= ['cart-svc', 'checkout-svc', 'edge-lb', 'payment-svc', 'search-svc'] fingerprints= 14
c-000-001 alerts= 2 services= ['notification-svc'] fingerprints= 2
c-000-002 alerts= 1 services= ['recommender-svc'] fingerprints= 1


In [80]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("=" * 60)
print("SAVED")
print("=" * 60)
print("Saved ->", OUTPUT_PATH)
print("Valid JSON:", OUTPUT_PATH.exists())

assert result["input_alerts"] > 0
assert result["output_clusters"] > 0
assert result["reduction_ratio"] >= 0.5
assert all("services" in c and "time_range" in c for c in result["clusters"])

print("Acceptance checks passed")


SAVED
Saved -> results/cluster_summary.json
Valid JSON: True
Acceptance checks passed


## Notes for SUBMIT.md

Gợi ý nội dung:

- `gap_sec = 120`: chọn 2 phút để gom alert burst trong cùng incident, tránh tách nhỏ khi alert xuất hiện lệch vài chục giây.
- `max_hop = 1`: chọn 1 hop để tránh over-correlation qua hub service. Nếu tăng lên 2, các service có thể bị gom quá rộng.
- Fingerprint không include `timestamp` hoặc `value` vì các field này thay đổi theo từng lần alert fire, làm mất tác dụng dedup.
- Limitation: topology grouping không phải root cause analysis. Hai alert gần nhau trên graph vẫn có thể độc lập nếu metric không liên quan hoặc workload khác nhau.
- Với 10000 alert, phần chậm là pairwise comparison trong mỗi session `O(n²)`. Có thể tối ưu bằng cách index theo service, metric family, hoặc xét theo connected components trước.
